In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
import zarr

DATA_ROOT = Path.home() / "ml-ds_data" / "input_data"
INPUT_FILE = DATA_ROOT / "2011.zarr"

In [ ]:
try:
    ds = xr.open_zarr(INPUT_FILE, consolidated=True)
except KeyError:
    print("Metadata not consolidated. Consolidating now...")
    zarr.consolidate_metadata(INPUT_FILE)
    ds = xr.open_zarr(INPUT_FILE, consolidated=True)

In [ ]:
ds

In [ ]:
ds.chunks

In [ ]:
time_index = pd.Index(ds.time.values)
print(time_index.has_duplicates)

In [ ]:
mean = ds.mean(dim=("time", "y", "x"))
std  = ds.std(dim=("time", "y", "x"))

In [ ]:
mean = mean.rename({v: f"{v}_mean" for v in mean.data_vars})
std  = std.rename({v: f"{v}_std"  for v in std.data_vars})

stats = xr.merge([mean, std])

In [ ]:
stats.to_zarr(DATA_ROOT / "era5_normalization_stats.zarr", mode="w")